# Masked Autoencoder — reconstruct what's hidden

**Foundation-model recipe** (the objective behind **MAE** and **VideoMAE**, lesson C3).

Hide most of an image's patches and train an encoder–decoder to reconstruct them. The model must learn structure to fill the gaps — a powerful self-supervised signal.

> CPU is fine.

In [ ]:
import os, torch, torch.nn as nn, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 1000)); S, P = 24, 4; NP = (S // P) ** 2; MASK = 0.6

## 1 · Synthetic images + patchify

In [ ]:
def gen(n):
    gx, gy = torch.meshgrid(torch.linspace(-1, 1, S), torch.linspace(-1, 1, S), indexing="ij")
    imgs = []
    for _ in range(n):
        cx, cy, r = torch.rand(1) - .5, torch.rand(1) - .5, 0.3 + 0.3 * torch.rand(1)
        imgs.append((((gx - cx) ** 2 + (gy - cy) ** 2) < r ** 2).float())
    return torch.stack(imgs).unsqueeze(1)
def patchify(x): return x.unfold(2, P, P).unfold(3, P, P).reshape(x.shape[0], NP, P * P)
def unpatchify(p):
    g = S // P
    return p.reshape(p.shape[0], g, g, P, P).permute(0, 1, 3, 2, 4).reshape(p.shape[0], 1, S, S)

## 2 · Encoder–decoder over patch tokens

In [ ]:
class MAE(nn.Module):
    def __init__(self, d=64):
        super().__init__(); self.emb = nn.Linear(P * P, d); self.pos = nn.Parameter(torch.randn(1, NP, d))
        self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(d, 4, 128, batch_first=True), 2)
        self.dec = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, P * P))
    def forward(self, patches, mask):
        tok = self.emb(patches) + self.pos
        tok = tok * (~mask).unsqueeze(-1)                       # zero out masked tokens
        return self.dec(self.enc(tok))
model = MAE().to(device); opt = torch.optim.Adam(model.parameters(), 1e-3)

## 3 · Pretrain — reconstruct the masked patches

In [ ]:
hist = []
for step in range(STEPS + 1):
    x = gen(64).to(device); p = patchify(x)
    mask = torch.rand(x.shape[0], NP, device=device) < MASK
    pred = model(p, mask); loss = (((pred - p) ** 2) * mask.unsqueeze(-1)).sum() / mask.sum() / (P * P)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % max(1, STEPS // 10) == 0: hist.append((step, round(loss.item(), 4))); print(f"step {step:4d}  masked recon MSE {loss.item():.4f}")

## 4 · Compare — input · masked · reconstruction

In [ ]:
x = gen(1).to(device); p = patchify(x); mask = torch.rand(1, NP, device=device) < MASK
with torch.no_grad(): pred = model(p, mask)
masked_in = unpatchify(p * (~mask).unsqueeze(-1)); recon = unpatchify(torch.where(mask.unsqueeze(-1), pred, p))
fig, ax = plt.subplots(1, 4, figsize=(11, 3))
for a, im, ttl in zip(ax, [x, masked_in, unpatchify(pred), recon], ["input", "masked", "decoded", "filled-in"]):
    a.imshow(im[0, 0].cpu().clamp(0, 1)); a.set_title(ttl); a.axis("off")
ax = plt.gcf().axes; plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/B_mae_pretrain/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/B_mae_pretrain"; os.makedirs(run, exist_ok=True)
torch.save(model.state_dict(), f"{run}/mae.pt")
json.dump({"recon_mse": hist}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## How this links to tracks A–D
The MAE objective underlies **C** VideoMAE and pretraining for **D** perception.

### Where to go next
- Extend masking across time → **VideoMAE** (tube masking, lesson C3).
- After pretraining, keep the encoder and fine-tune it for a downstream task.